# eCommerce Funnel Analysis
**Goal:** Calculate unique user conversion rates from View -> Intent -> Purchase.

### How to Use This Pipeline
This notebook is designed as an automated ETL pipeline. 
1. Place the new month's raw CSV file into the `data/raw/` folder.
2. In the second code cell, update the `pd.read_csv()` path to point to the new file.
3. Click **Restart Kernel and Run All Cells**.
4. The calculated funnel CSVs will automatically update in the `data/processed/` folder.

In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option('display.max_columns', None)

In [27]:
#df = pd.read_csv("../data/raw/2019-Nov.csv")
#df.head()

In [28]:
#Defining only the columns we actually need for the funnel
needed_columns = ['event_type', 'user_id', 'brand']

# 2.optimize the text columns into memory-saving categories
optimized_types = {
    'event_type': 'category', 
    'brand': 'category'
}

# 3. Load the data using our optimizations
df = pd.read_csv(
    "../data/raw/2019-Nov.csv", 
    usecols=needed_columns,
    dtype=optimized_types
)

df.head()

,event_type,brand,user_id
0,view,xiaomi,520088904
1,view,janome,530496790
2,view,creed,561587266
3,view,lg,518085591
4,view,xiaomi,558856683


## Step 1: Raw Event Frequencies
First, looking at the total number of actions taken across the platform. However, these are total events, not unique users.

In [29]:
event_count = df['event_type'].value_counts()
event_count

event_type
view        63556110
cart         3028930
purchase      916939
Name: count, dtype: int64

## Step 2: Unique Users per Event
To build an accurate funnel, we calculate the distinct number of users at each stage. Notice that 'purchase' is higher than 'cart', indicating a direct checkout feature.

In [30]:
distinct_event_count = df.groupby('event_type')['user_id'].nunique()
distinct_event_count

event_type
cart         826323
purchase     441638
view        3695598
Name: user_id, dtype: int64

## Step 3: Funnel Stage Deduplication (The Intent Bucket)
Because users can bypass the cart, we create a unified 'Intent' stage (combining 'cart' and 'purchase'). We use `.nunique()` on this subset to ensure users who both carted and purchased are not counted twice.

In [31]:
intent_data = df[df['event_type'].isin(['cart','purchase'])]
true_intent_users = intent_data['user_id'].nunique()
true_intent_users


867914

## Step 4: Dynamic Funnel Aggregation
Finally, assembling these deduplicated metrics into a clean, automated summary table for dashboard ingestion.

In [32]:
data = {
    "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
    "Unique_Users": [
        distinct_event_count['view'],      
        true_intent_users,                 
        distinct_event_count['purchase']   
    ]
}

funnel_df = pd.DataFrame(data)
funnel_df

,Funnel_Stage,Unique_Users
0,01_View,3695598
1,02_Intent,867914
2,03_Purchase,441638


## Step 5: Conversion and Drop-off Analysis
To fulfill the project requirements, we calculate the stage-to-stage conversion and drop-off rates using vectorized division and the `.shift()` method to align our rows.


In [33]:
funnel_df['conversion_rate%'] = (funnel_df['Unique_Users'] / funnel_df['Unique_Users'].shift(1)) * 100 
funnel_df['conversion_rate%'] = funnel_df['conversion_rate%'].fillna(100).round(2)
funnel_df['drop_off_rate%'] = 100 - funnel_df['conversion_rate%']  
funnel_df

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,3695598,100.00,0.00
1,02_Intent,867914,23.49,76.51
2,03_Purchase,441638,50.88,49.12


### Key Insights & Business Recommendations:
* **The Massive Top-of-Funnel Leak:** Historically, the vast majority of traffic bounces after seeing the product page. Refer to the output table below for this month's exact View-to-Intent drop-off rate. 
    * **Recommendation:** The marketing and UI teams should continuously monitor this drop-off. Consider A/B testing better product images or making the "Add to Cart" button more prominent.
* **The High-Intent Goldmine:** Once a user shows intent (cart or checkout), the conversion rate to purchase is consistently high. See the Intent-to-Purchase conversion rate below.

## Step 6: Scaling Analysis with Functions (DRY Principle)
To compare performance across different segments (e.g., brands), we wrap our funnel logic into a reusable Python function. This prevents code duplication and allows us to generate custom funnels dynamically.

In [34]:
def analyze_brand_funnel(brand_name):
    # 1. Isolating the brand
    brand_df = df[df['brand'] == brand_name]
    
    # 2. Getting distinct counts per stage
    distinct_counts = brand_df.groupby('event_type')['user_id'].nunique()
    
    # Calculating deduplicated intent
    true_intent = brand_df[brand_df['event_type'].isin(['cart', 'purchase'])]['user_id'].nunique()
    
    # 4. dataframe
    data = {
        "Funnel_Stage": ["01_View", "02_Intent", "03_Purchase"],
        "Unique_Users": [
            distinct_counts.get('view', 0),    
            true_intent,
            distinct_counts.get('purchase', 0) 
        ]
    }
    
    brand_funnel_df = pd.DataFrame(data)
    
    # 5. Adding the columns
    brand_funnel_df['conversion_rate%'] = (brand_funnel_df['Unique_Users'] / brand_funnel_df['Unique_Users'].shift(1)) * 100
    brand_funnel_df['conversion_rate%'] = brand_funnel_df['conversion_rate%'].fillna(100).round(2)
    brand_funnel_df['drop_off_rate%'] = 100 - brand_funnel_df['conversion_rate%']
    
    return brand_funnel_df

## Step 7: Brand Comparison (Apple vs. Samsung)
Testing our function to compare the highest volume brands. 

**Key Business Insight:** 
Look at the output table below to compare how rivals like Apple and Samsung are performing this month. Typically, one brand will have a stronger top-of-funnel draw, while the other might have higher checkout closing power.

In [35]:
analyze_brand_funnel('apple')

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,921271,100.00,0.00
1,02_Intent,165962,18.01,81.99
2,03_Purchase,83664,50.41,49.59


In [36]:
analyze_brand_funnel('samsung')

,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,01_View,1174393,100.00,0.00
1,02_Intent,216112,18.40,81.60
2,03_Purchase,113477,52.51,47.49


## Step 8: Automated Top 10 Brand Pipeline (Dashboard Prep)
Hardcoding brand names does not scale for monthly reporting. To prepare this data for a BI dashboard (like Power BI or Tableau), I programmatically identified the highest-volume brands and format their funnels into a single, stacked table ("Long Data").

**The Pipeline Logic:**
1. **Dynamic Identification:** We calculate the top 10 brands by total event volume.
2. **Loop & Append:** We loop each top brand through our `analyze_brand_funnel` function, inserting a `Brand` column so the BI tool can filter the data.
3. **Stacking (Concat):** We combine all 10 individual tables into one master dataframe (`master_brand_df`).
4. **Export:** Both the overall funnel and the top 10 brand comparison tables are exported as clean CSVs to the `data/processed/` directory, ready for dashboard ingestion.

In [37]:
#Exporting the overall funnel
funnel_df.to_csv('../data/processed/01_overall_funnel_NOV.csv', index=False)

# Finding the top 10 brands dynamically based on who has the most rows
top_10_brands = df['brand'].value_counts().head(10).index.tolist()

# Createing an empty list to hold our tables
all_brand_funnels = []

# 4. Loop through the top 10 brands
for brand in top_10_brands:
    # Run function
    temp_df = analyze_brand_funnel(brand)
    
    # Add a column so we know which brand
    temp_df.insert(0, 'Brand', brand) 
    
    # Add this brand's table to our list
    all_brand_funnels.append(temp_df)

# 5. Stack all 10 tables on top of each other into one master dataframe
master_brand_df = pd.concat(all_brand_funnels, ignore_index=True)

# 6. Export the final master brand table to CSV!
master_brand_df.to_csv('../data/processed/02_brand_comparison_NOV.csv', index=False)

# Display the master table to check our work
master_brand_df

,Brand,Funnel_Stage,Unique_Users,conversion_rate%,drop_off_rate%
0,samsung,01_View,1174393,100.00,0.00
1,samsung,02_Intent,216112,18.40,81.60
2,samsung,03_Purchase,113477,52.51,47.49
3,apple,01_View,921271,100.00,0.00
4,apple,02_Intent,165962,18.01,81.99
5,apple,03_Purchase,83664,50.41,49.59
6,xiaomi,01_View,610454,100.00,0.00
7,xiaomi,02_Intent,95991,15.72,84.28
8,xiaomi,03_Purchase,43988,45.83,54.17
9,huawei,01_View,285037,100.00,0.00
